# LeetCode #1092: Shortest Common Supersequence

https://leetcode.com/problems/shortest-common-supersequence/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^{m+n})$ | $O(m+n)$ |
| **Optimal: DP (LCS) + Reconstruction ★** | $O(m \cdot n)$ | $O(m \cdot n)$ |

---

## Understanding the Methods

### Brute Force
Try all interleavings of the two strings and check if each is a supersequence of both. Exponential in $m + n$.

### Optimal: DP (LCS) + Reconstruction ★
The shortest common supersequence (SCS) has length $m + n - \text{LCS}(s1, s2)$. Build the standard LCS DP table, then traceback: wherever LCS characters match both strings, include the character once; otherwise include whichever string's character was chosen to extend the LCS. This produces the actual SCS string, not just its length, in $O(m \cdot n)$ time and space.

**Constraints:**
* 1 <= str1.length, str2.length <= 1000
* str1 and str2 consist of lowercase English letters

## Solutions
### C#

In [ ]:
// DP (LCS) + traceback reconstruction of the shortest common supersequence
public class Solution {
    public string ShortestCommonSupersequence(string str1, string str2) {
        int m = str1.Length, n = str2.Length;
        // Build LCS DP table
        int[,] dp = new int[m + 1, n + 1];
        for (int i = 1; i <= m; i++)
            for (int j = 1; j <= n; j++)
                dp[i, j] = str1[i - 1] == str2[j - 1]
                    ? dp[i - 1, j - 1] + 1
                    : Math.Max(dp[i - 1, j], dp[i, j - 1]);

        // Traceback: build SCS from both strings using the LCS table
        var sb = new System.Text.StringBuilder();
        int r = m, c = n;
        while (r > 0 && c > 0) {
            if (str1[r - 1] == str2[c - 1]) {
                // LCS character — include it once and consume from both strings
                sb.Append(str1[r - 1]);
                r--; c--;
            } else if (dp[r - 1, c] > dp[r, c - 1]) {
                // Coming from above: include str1's character
                sb.Append(str1[r - 1]);
                r--;
            } else {
                // Coming from the left: include str2's character
                sb.Append(str2[c - 1]);
                c--;
            }
        }
        // Append any remaining characters from either string
        while (r > 0) sb.Append(str1[--r]);
        while (c > 0) sb.Append(str2[--c]);

        // Result was built in reverse
        var arr = sb.ToString().ToCharArray();
        Array.Reverse(arr);
        return new string(arr);
    }
}

### Python

In [ ]:
# DP (LCS) + traceback reconstruction of the shortest common supersequence
class Solution:
    def shortestCommonSupersequence(self, str1: str, str2: str) -> str:
        m, n = len(str1), len(str2)
        # Build LCS DP table
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if str1[i - 1] == str2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

        # Traceback to reconstruct the SCS
        result = []
        r, c = m, n
        while r > 0 and c > 0:
            if str1[r - 1] == str2[c - 1]:
                # LCS match: include the character once from both strings
                result.append(str1[r - 1])
                r -= 1; c -= 1
            elif dp[r - 1][c] > dp[r][c - 1]:
                # LCS came from str1 — include str1's char
                result.append(str1[r - 1])
                r -= 1
            else:
                # LCS came from str2 — include str2's char
                result.append(str2[c - 1])
                c -= 1
        # Drain remaining characters from whichever string is not exhausted
        while r > 0:
            result.append(str1[r - 1]); r -= 1
        while c > 0:
            result.append(str2[c - 1]); c -= 1

        return ''.join(reversed(result))

### Go

In [ ]:
// DP (LCS) + traceback reconstruction of the shortest common supersequence
package main

func shortestCommonSupersequence(str1 string, str2 string) string {
    m, n := len(str1), len(str2)
    // Build LCS DP table
    dp := make([][]int, m+1)
    for i := range dp {
        dp[i] = make([]int, n+1)
    }
    for i := 1; i <= m; i++ {
        for j := 1; j <= n; j++ {
            if str1[i-1] == str2[j-1] {
                dp[i][j] = dp[i-1][j-1] + 1
            } else if dp[i-1][j] > dp[i][j-1] {
                dp[i][j] = dp[i-1][j]
            } else {
                dp[i][j] = dp[i][j-1]
            }
        }
    }

    // Traceback to build the SCS in reverse
    result := make([]byte, 0, m+n)
    r, c := m, n
    for r > 0 && c > 0 {
        if str1[r-1] == str2[c-1] {
            // LCS match: consume from both, include once
            result = append(result, str1[r-1])
            r--; c--
        } else if dp[r-1][c] > dp[r][c-1] {
            result = append(result, str1[r-1])
            r--
        } else {
            result = append(result, str2[c-1])
            c--
        }
    }
    for r > 0 { result = append(result, str1[r-1]); r-- }
    for c > 0 { result = append(result, str2[c-1]); c-- }

    // Reverse the accumulated bytes to get the correct order
    for i, j := 0, len(result)-1; i < j; i, j = i+1, j-1 {
        result[i], result[j] = result[j], result[i]
    }
    return string(result)
}

### Rust

In [ ]:
// DP (LCS) + traceback reconstruction of the shortest common supersequence
impl Solution {
    pub fn shortest_common_supersequence(str1: String, str2: String) -> String {
        let s1 = str1.as_bytes();
        let s2 = str2.as_bytes();
        let (m, n) = (s1.len(), s2.len());

        // Build LCS DP table
        let mut dp = vec![vec![0usize; n + 1]; m + 1];
        for i in 1..=m {
            for j in 1..=n {
                dp[i][j] = if s1[i - 1] == s2[j - 1] {
                    dp[i - 1][j - 1] + 1
                } else {
                    dp[i - 1][j].max(dp[i][j - 1])
                };
            }
        }

        // Traceback to reconstruct the SCS in reverse
        let mut result: Vec<u8> = Vec::with_capacity(m + n);
        let (mut r, mut c) = (m, n);
        while r > 0 && c > 0 {
            if s1[r - 1] == s2[c - 1] {
                // LCS character: include once and advance both pointers
                result.push(s1[r - 1]);
                r -= 1; c -= 1;
            } else if dp[r - 1][c] > dp[r][c - 1] {
                result.push(s1[r - 1]); r -= 1;
            } else {
                result.push(s2[c - 1]); c -= 1;
            }
        }
        while r > 0 { result.push(s1[r - 1]); r -= 1; }
        while c > 0 { result.push(s2[c - 1]); c -= 1; }

        result.reverse();
        String::from_utf8(result).unwrap()
    }
}

## Example Scenarios

**1. Common Case** — Two short overlapping strings

**Input:** `str1 = "abac", str2 = "cab"`
LCS("abac","cab") = "ab" (length 2). SCS length = 4+3−2 = 5. Traceback weaves in both strings around the LCS: "cabac". All characters of both inputs appear as subsequences.

**2. Slightly Complex** — Shared prefix

**Input:** `str1 = "abc", str2 = "abcd"`
LCS = "abc" (length 3). SCS = str1 + remaining str2 = "abcd". The traceback drains str1 entirely matching str2's first three chars, then appends 'd'. Result: `"abcd"`.

**3. Edge Case: Time Factor** — No common characters

**Input:** `str1 = "aaa...a"` (1000 'a'), `str2 = "bbb...b"` (1000 'b')
LCS = "" (length 0). SCS = str1 + str2 (length 2000). The DP table is filled as 0 everywhere; traceback drains str2 then str1. The DP fill is the bottleneck at $O(m \cdot n) = O(10^6)$.

**4. Edge Case: Space Factor** — Maximum inputs (1000 × 1000)

**Input:** `str1.length = str2.length = 1000`
The DP table holds $1001 \times 1001 \approx 10^6$ integers — the dominant space cost at $O(m \cdot n)$. The output string is at most 2000 characters.

**5. Almost-Impossible but Plausible** — Strings are each other's reverse

**Input:** `str1 = "abcde", str2 = "edcba"`
LCS = 1 (any single character like 'a' or 'e'). SCS length = 5+5−1 = 9. The traceback must include every character of both strings with only one overlap. Result length is exactly 9, confirming the formula $m+n-\text{LCS}$.